# Results

In [1]:
import torch
import timm
import tome
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from eval import evaluate
from SReT import SReT_T_distill
import SReT_ToMe

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Current GPU Device: {torch.cuda.current_device()}")

batch_size = 128
dataset_dir = '/media/datasets/imagenet/val'
dataset_transform = transforms.Compose([
    transforms.Resize(256), 
    transforms.CenterCrop(224), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(dataset_dir, transform=dataset_transform)
dataset_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
print(f'Loaded {len(dataset)} ImageNet-1k Validation images')

CUDA available: True
GPU Device Name: NVIDIA GeForce RTX 4060 Ti
Current GPU Device: 0
Loaded 50000 ImageNet-1k Validation images


## Evaluate

#### DeiT-Tiny-Distill + ToMe

In [4]:
model = timm.create_model("deit_tiny_distilled_patch16_224", pretrained=True)
tome.patch.timm(model, prop_attn=True)
model = model.cuda().eval()

rates = [0, 10, 15, 20]

for r in rates:
    print(f"--- r = {r} ---")
    model.r = r
    _ = evaluate(model, dataset_loader, batch_size)

--- r = 0 ---

--- Tracking Token Sequence Length ---
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
[ToMeBlock] Tokens: 198 -> 198
-----------------------------------------------

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         74.40 %
 Total Parameters:       5.91 M
 Theoretical FLOPs:      2.17 G
 Throughput (BS=128):    1776.15 images/sec
 Throughput (BS=64):    1904.04 images/sec
 Throughput (BS=32):    1966.15 images/sec
 Throughput (BS=16):    2392.25 images/sec
 Throughput (BS=1):    661.86 images/sec
 Model Weights VRAM:     106.76 MB
 Peak Activation Memory: 247.01 MB

--- r = 10 ---

--- Tracking Token Sequence Length ---
[ToMeBlo

#### PiT

In [2]:
# tiny
model = timm.create_model("pit_ti_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)


--- Tracking Token Sequence Length ---
[Block] Tokens: 731 -> 731
[Block] Tokens: 731 -> 731
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 51 -> 51
[Block] Tokens: 51 -> 51
[Block] Tokens: 51 -> 51
[Block] Tokens: 51 -> 51
-----------------------------------------------

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         74.42 %
 Total Parameters:       5.10 M
 Theoretical FLOPs:      1.01 G
 Throughput (BS=128):    1796.78 images/sec
 Throughput (BS=64):    1849.91 images/sec
 Throughput (BS=32):    1943.30 images/sec
 Throughput (BS=16):    2064.07 images/sec
 Throughput (BS=1):    679.02 images/sec
 Model Weights VRAM:     103.66 MB
 Peak Activation Memory: 1203.36 MB



In [3]:
# xs
model = timm.create_model("pit_xs_distilled_224", pretrained=True)
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)


--- Tracking Token Sequence Length ---
[Block] Tokens: 731 -> 731
[Block] Tokens: 731 -> 731
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 198 -> 198
[Block] Tokens: 51 -> 51
[Block] Tokens: 51 -> 51
[Block] Tokens: 51 -> 51
[Block] Tokens: 51 -> 51
-----------------------------------------------

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         79.13 %
 Total Parameters:       11.00 M
 Theoretical FLOPs:      2.21 G
 Throughput (BS=128):    1360.63 images/sec
 Throughput (BS=64):    1407.75 images/sec
 Throughput (BS=32):    1446.99 images/sec
 Throughput (BS=16):    1544.74 images/sec
 Throughput (BS=1):    664.78 images/sec
 Model Weights VRAM:     126.24 MB
 Peak Activation Memory: 1283.28 MB



#### EfficientViT-M2

In [2]:
efficientvit_model = timm.create_model('efficientvit_m2.r224_in1k', pretrained=True)
efficientvit_model = efficientvit_model.cuda().eval()
_ = evaluate(efficientvit_model, dataset_loader, batch_size)


--- Tracking Token Sequence Length ---
[EfficientVitBlock] Tokens: 128 -> 128
[EfficientVitBlock] Tokens: 192 -> 192
[EfficientVitBlock] Tokens: 192 -> 192
[EfficientVitBlock] Tokens: 224 -> 224
[EfficientVitBlock] Tokens: 224 -> 224
[EfficientVitBlock] Tokens: 224 -> 224
-----------------------------------------------

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         70.61 %
 Total Parameters:       4.19 M
 Theoretical FLOPs:      0.41 G
 Throughput (BS=128):    7646.66 images/sec
 Throughput (BS=64):    8101.10 images/sec
 Throughput (BS=32):    7777.50 images/sec
 Throughput (BS=16):    4590.95 images/sec
 Throughput (BS=1):    296.42 images/sec
 Model Weights VRAM:     99.61 MB
 Peak Activation Memory: 195.51 MB



#### SReT-Tiny-Distill

In [4]:
model = SReT_T_distill(pretrained=False) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)


--- Tracking Token Sequence Length ---
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
-----------------------------------------------

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         77.39 %
 Total Parameters:       4.76 

#### SReT + ToMe
- Constant token reduction

In [3]:
rates = [0, 10, 15, 20]

for r in rates:
    print(f"--- r = {r} ---")
    model = SReT_ToMe.SReT_T_distill(pretrained=False, constant_r=r) # initialize
    checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
    model.load_state_dict(checkpoint['model'])
    model = model.cuda().eval()
    _ = evaluate(model, dataset_loader, batch_size)

--- r = 0 ---

--- Tracking Token Sequence Length ---
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 784 -> 784
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 196 -> 196
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
[Transformer_Block] Tokens: 49 -> 49
-----------------------------------------------

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         77.44 %
 Total Parameter

#### SReT + ToMe
- Dynamic Decaying Schedule
- Best result from grid search results (r=0.3, alpha=0.1)

In [2]:
model = SReT_ToMe.SReT_T_distill(pretrained=False, initial_r_ratio=0.3, alpha=0.1) # initialize
checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.cuda().eval()
_ = evaluate(model, dataset_loader, batch_size)


--- Tracking Token Sequence Length ---
[Transformer_Block] Tokens: 784 -> 544
[Transformer_Block] Tokens: 544 -> 520
[Transformer_Block] Tokens: 520 -> 512
[Transformer_Block] Tokens: 512 -> 512
[Transformer_Block] Tokens: 196 -> 136
[Transformer_Block] Tokens: 136 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 128 -> 128
[Transformer_Block] Tokens: 49 -> 35
[Transformer_Block] Tokens: 35 -> 34
[Transformer_Block] Tokens: 34 -> 34
[Transformer_Block] Tokens: 34 -> 34
[Transformer_Block] Tokens: 34 -> 34
[Transformer_Block] Tokens: 34 -> 34
-----------------------------------------------

 Target Batch Size:      128
--------------------------------------------------
 Top-1 Accuracy:         74.90 %
 Total Parameters:       4.76 